# Banco de Dados do Projeto

Neste notebook é construída a camada de armazenamento de dados do projeto utilizando **SQLite**, um sistema de gerenciamento de banco de dados relacional leve e open source.

O objetivo desta etapa é estruturar os dados tratados no processo de ETL em um banco relacional, permitindo:

- Persistência dos dados processados
- Organização das informações em uma estrutura tabular
- Realização de consultas analíticas utilizando **SQL**
- Integração futura com aplicações como **dashboards e sistemas inteligentes**

Essa abordagem também segue boas práticas de **engenharia de dados**, separando as etapas de processamento, armazenamento e análise.

In [17]:
import sqlite3
import pandas as pd

In [18]:
# Nome do banco
db_name = "../database/hospital.db"

conn = sqlite3.connect(db_name)
cursor = conn.cursor()
print("Banco conectado com sucesso")

# Leitura
df = pd.read_csv("../data/processed/kaggle_tratado.csv")

# Remover tabela antiga
cursor.execute("DROP TABLE IF EXISTS consultas")

# Criar nova tabela com colunas corretas
cursor.execute("""
CREATE TABLE consultas (
    
    data_agendamento TEXT,
    data_consulta TEXT,
    
    idade INTEGER,
    
    bolsa_familia INTEGER,
    hipertensao INTEGER,
    diabetes INTEGER,
    alcoolismo INTEGER,
    deficiencia INTEGER,
    
    sms_recebido INTEGER,
    
    dias_espera INTEGER,
    
    dia_semana_consulta INTEGER,
    mes_consulta INTEGER,
    
    consulta_mesmo_dia INTEGER,
    
    no_show INTEGER
    
)
""")

# Inserir no banco
df[[
    "data_agendamento",
    "data_consulta",
    "idade",
    "bolsa_familia",
    "hipertensao",
    "diabetes",
    "alcoolismo",
    "deficiencia",
    "sms_recebido",
    "dias_espera",
    "dia_semana_consulta",
    "mes_consulta",
    "consulta_mesmo_dia",
    "no_show"
]].to_sql(
    "consultas",
    conn,
    if_exists="append",
    index=False
)

# Finalizar
query = "SELECT COUNT(*) FROM consultas"

pd.read_sql(query, conn)

# Amostra
print("Amostra da tabela consultas:")

for row in cursor.execute("SELECT * FROM consultas LIMIT 5"):
    print(row)

# Fechar banco
conn.commit()
conn.close()

print("Banco salvo com sucesso")

Banco conectado com sucesso
Amostra da tabela consultas:
('2016-04-29', '2016-04-29', 62, 0, 1, 0, 0, 0, 0, 0, 4, 4, 1, 0)
('2016-04-29', '2016-04-29', 56, 0, 0, 0, 0, 0, 0, 0, 4, 4, 1, 0)
('2016-04-29', '2016-04-29', 62, 0, 0, 0, 0, 0, 0, 0, 4, 4, 1, 0)
('2016-04-29', '2016-04-29', 8, 0, 0, 0, 0, 0, 0, 0, 4, 4, 1, 0)
('2016-04-29', '2016-04-29', 56, 0, 1, 1, 0, 0, 0, 0, 4, 4, 1, 0)
Banco salvo com sucesso


## Análises exploratórias utilizando SQL

#### Taxa Geral de Faltas em Consultas

Uma das principais métricas analisadas no projeto é a taxa de faltas em consultas médicas.

Essa métrica é importante para hospitais e clínicas, pois faltas impactam diretamente:

- planejamento de recursos
- disponibilidade de profissionais
- eficiência do atendimento

A consulta abaixo calcula a taxa percentual de faltas.

In [19]:
# Abrir conexão
conn = sqlite3.connect("../database/hospital.db")

In [20]:

query = """
SELECT
AVG(no_show) * 100 AS taxa_faltas_percentual
FROM consultas
"""

pd.read_sql(query, conn)

,taxa_faltas_percentual
0,20.189828


### Análise de Faltas por Mês

Nesta análise buscamos verificar se existe variação na taxa de faltas dependendo do mês da consulta.

In [21]:
query = """
SELECT
mes_consulta,
AVG(no_show) * 100 AS taxa_faltas
FROM consultas
GROUP BY mes_consulta
ORDER BY mes_consulta
"""

pd.read_sql(query, conn)

,mes_consulta,taxa_faltas
0,4,19.567233
1,5,20.781582
2,6,18.457467


### Impacto do Envio de SMS no Comparecimento

Diversos estudos mostram que lembretes de consultas podem reduzir faltas.

Nesta análise verificamos se pacientes que receberam SMS apresentaram menor taxa de ausência.

In [22]:
query = """
SELECT
sms_recebido,
AVG(no_show) * 100 AS taxa_faltas
FROM consultas
GROUP BY sms_recebido
"""

pd.read_sql(query, conn)

,sms_recebido,taxa_faltas
0,0,16.697984
1,1,27.574545


### Taxa de Faltas por Faixa Etária

A análise por faixa etária pode ajudar a identificar grupos de pacientes com maior probabilidade de ausência.

In [23]:
query = """
SELECT
CASE
WHEN idade < 18 THEN '0-17'
WHEN idade < 40 THEN '18-39'
WHEN idade < 60 THEN '40-59'
ELSE '60+'
END AS faixa_etaria,

AVG(no_show) * 100 AS taxa_faltas

FROM consultas

GROUP BY faixa_etaria
ORDER BY faixa_etaria
"""

pd.read_sql(query, conn)

,faixa_etaria,taxa_faltas
0,0-17,21.900796
1,18-39,23.264052
2,40-59,18.808194
3,60+,15.307954


In [24]:
# Fechar banco
conn.close()

## Conclusão

Neste notebook foi implementada a camada de armazenamento de dados do projeto utilizando SQLite.

As principais atividades realizadas foram:

- criação do banco de dados
- estruturação da tabela de consultas
- inserção dos dados tratados
- realização de análises exploratórias utilizando SQL

Essa etapa contribui para a organização da arquitetura de dados do projeto e possibilita análises adicionais diretamente sobre o banco.

Além disso, essa base poderá ser utilizada futuramente por aplicações como:

- dashboards interativos
- modelos de machine learning
- assistentes inteligentes baseados em IA